In [45]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
# from prophet import Prophet

In [46]:
# Loading the fao dataset to be used for predicting yield for different crops
fao_final_df = pd.read_csv('faostat_model_dataset_clean.csv')
fao_final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 34 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   area                                    252 non-null    str    
 1   item                                    252 non-null    str    
 2   year                                    252 non-null    int64  
 3   domestic_supply_quantity                252 non-null    float64
 4   export_quantity                         251 non-null    float64
 5   feed                                    113 non-null    float64
 6   food                                    252 non-null    float64
 7   import_quantity                         244 non-null    float64
 8   losses                                  252 non-null    float64
 9   other_uses_non_food                     76 non-null     float64
 10  processing                              117 non-null    float64
 11  prod

In [47]:
yield_df_final = pd.read_csv('Data\yield_cleaned.csv')
yield_df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4396 entries, 0 to 4395
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   item             4396 non-null   str    
 1   year             4396 non-null   int64  
 2   unit             4396 non-null   str    
 3   yield_kg_per_ha  4396 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 137.5 KB


In [48]:
fao_final_df = fao_final_df.merge(
    yield_df_final,
    on=['item', 'year'],
    how='left'
)

print("Final dataset shape:", fao_final_df.shape)

print("\nMissing values (top 15):")
print(fao_final_df.isna().sum().sort_values(ascending=False).head(15))

print("\nSample:")
print(fao_final_df.head())

Final dataset shape: (252, 36)

Missing values (top 15):
yield_kg_per_ha             224
unit                        224
other_uses_non_food         176
feed                        139
processing                  135
seed                        131
stock_variation              32
import_quantity               8
export_quantity               1
year                          0
item                          0
area                          0
domestic_supply_quantity      0
losses                        0
production                    0
dtype: int64

Sample:
    area     item  year  domestic_supply_quantity  export_quantity  feed  \
0  Kenya  Bananas  2010                    1583.0              0.0   NaN   
1  Kenya  Bananas  2011                    1227.0              0.0   NaN   
2  Kenya  Bananas  2012                    1208.0              0.0   NaN   
3  Kenya  Bananas  2013                    1375.0              0.0   NaN   
4  Kenya  Bananas  2014                    1645.0            

In [49]:
# Save final FAOSTAT modelling dataframe after yield merge
fao_final_df.to_csv("Data/fao_final_df_with_yield.csv", index=False)

print("Saved successfully")
print(fao_final_df.shape)

Saved successfully
(252, 36)


In [50]:
# Check yield availability by crop
yield_check = (
    fao_final_df
    .groupby('item')
    .agg(
        total_rows=('year', 'count'),
        yield_rows=('yield_kg_per_ha', 'count'),
        min_year=('year', 'min'),
        max_year=('year', 'max')
    )
    .sort_values('yield_rows', ascending=False)
)

print(yield_check)

                            total_rows  yield_rows  min_year  max_year
item                                                                  
Bananas                             14          14      2010      2023
Sweet potatoes                      14          14      2010      2023
Barley and products                 14           0      2010      2023
Cassava and products                14           0      2010      2023
Coffee and products                 14           0      2010      2023
Coconuts - Incl Copra               14           0      2010      2023
Lemons, Limes and products          14           0      2010      2023
Maize and products                  14           0      2010      2023
Millet and products                 14           0      2010      2023
Groundnuts                          14           0      2010      2023
Onions                              14           0      2010      2023
Oranges, Mandarines                 14           0      2010      2023
Potato

In [51]:
# Show only crops with actual yield values
crops_with_yield = yield_check[yield_check['yield_rows'] > 0]

print(crops_with_yield)

                total_rows  yield_rows  min_year  max_year
item                                                      
Bananas                 14          14      2010      2023
Sweet potatoes          14          14      2010      2023


# Confirm mismatch

In [52]:
# Crops in main dataset
main_items = set(fao_final_df['item'].unique())

# Crops in yield dataset
yield_items = set(yield_df_final['item'].unique())

print("Items in main but NOT in yield:")
print(main_items - yield_items)

print("\nItems in yield but NOT in main:")
print(yield_items - main_items)

Items in main but NOT in yield:
{'Cassava and products', 'Coffee and products', 'Lemons, Limes and products', 'Oranges, Mandarines', 'Potatoes and products', 'Maize and products', 'Tomatoes and products', 'Millet and products', 'Barley and products', 'Sorghum and products', 'Groundnuts', 'Pineapples and products', 'Coconuts - Incl Copra', 'Onions', 'Wheat and products', 'Rice and products'}

Items in yield but NOT in main:
{'Other beans, green', 'Pepper (Piper spp.), raw', 'Potatoes', 'Soya beans', 'Barley', 'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)', 'Sorghum', 'Millet', 'Other nuts (excluding wild edible nuts and groundnuts), in shell, n.e.c.', 'Nutmeg, mace, cardamoms, raw', 'Lemons and limes', 'Peaches and nectarines', 'Sesame seed', 'Tea leaves', 'Asparagus', 'Plums and sloes', 'Seed cotton, unginned', 'Wheat', 'Lettuce and chicory', 'Cauliflowers and broccoli', 'Cloves (whole stems), raw', 'Abaca, manila hemp, raw', 'Vanilla, raw', 'Oats', 'Avocados', 'Pomelos

## create a mapping dictionary

In [53]:
# ============================================================
# STEP 1: MAP YIELD CROP NAMES TO FAOSTAT FOOD BALANCE NAMES
# ============================================================
# The yield dataset uses specific crop names, while the food balance
# dataset uses broader FAOSTAT categories such as "Maize and products".
#
# This dictionary tells Python which yield crop corresponds to which
# food balance crop category.

yield_to_fao_mapping = {
    'Maize (corn)': 'Maize and products',
    'Rice': 'Rice and products',
    'Cassava, fresh': 'Cassava and products',
    'Sorghum': 'Sorghum and products',
    'Millet': 'Millet and products',
    'Barley': 'Barley and products',
    'Wheat': 'Wheat and products',
    'Potatoes': 'Potatoes and products',
    'Tomatoes': 'Tomatoes and products',
    'Pineapples': 'Pineapples and products',
    'Coffee, green': 'Coffee and products',
    'Coconuts, in shell': 'Coconuts - Incl Copra',
    'Groundnuts, excluding shelled': 'Groundnuts',
    'Onions and shallots, dry (excluding dehydrated)': 'Onions',
    'Oranges': 'Oranges, Mandarines',
    'Tangerines, mandarins, clementines': 'Oranges, Mandarines',
    'Lemons and limes': 'Lemons, Limes and products',
    'Plantains and cooking bananas': 'Bananas',
    'Sweet potatoes': 'Sweet potatoes'
}

# Create a new mapped item column in the yield dataset
yield_df_final['item_mapped'] = yield_df_final['item'].map(yield_to_fao_mapping)

# Check how many yield rows are now mapped
print("Mapped yield rows:", yield_df_final['item_mapped'].notna().sum())
print("Unmapped yield rows:", yield_df_final['item_mapped'].isna().sum())

print("\nMapped crop names:")
print(
    yield_df_final[['item', 'item_mapped']]
    .dropna()
    .drop_duplicates()
    .sort_values('item_mapped')
)

Mapped yield rows: 1175
Unmapped yield rows: 3221

Mapped crop names:
                                                 item  \
3080                    Plantains and cooking bananas   
370                                            Barley   
660                                    Cassava, fresh   
1005                               Coconuts, in shell   
1069                                    Coffee, green   
1431                    Groundnuts, excluding shelled   
1534                                 Lemons and limes   
1714                                     Maize (corn)   
1842                                           Millet   
2050  Onions and shallots, dry (excluding dehydrated)   
3986               Tangerines, mandarins, clementines   
2114                                          Oranges   
3016                                       Pineapples   
3241                                         Potatoes   
3373                                             Rice   
3629              

In [54]:
# ============================================================
# STEP 2: RE-MERGE YIELD DATA USING THE MAPPED CROP NAMES
# ============================================================
# Previously, we merged on:
#   item + year
#
# That failed for most crops because names were different:
#   "Maize and products" vs "Maize (corn)"
#
# Now we merge using:
#   fao_final_df['item'] == yield_df_final['item_mapped']
#   and year == year

# First, remove the old yield columns if they already exist
cols_to_drop = ['yield_kg_per_ha', 'unit']

fao_final_df_clean = fao_final_df.drop(
    columns=[col for col in cols_to_drop if col in fao_final_df.columns]
)

# Keep only mapped yield rows and the columns needed for merge
yield_mapped = (
    yield_df_final
    .dropna(subset=['item_mapped'])
    [['item_mapped', 'year', 'unit', 'yield_kg_per_ha']]
    .drop_duplicates(subset=['item_mapped', 'year'])
)

# Merge correctly
fao_final_df = fao_final_df_clean.merge(
    yield_mapped,
    left_on=['item', 'year'],
    right_on=['item_mapped', 'year'],
    how='left'
)

# Drop helper merge column
fao_final_df = fao_final_df.drop(columns=['item_mapped'])

print("Final dataset shape:", fao_final_df.shape)

print("\nMissing yield values:")
print(fao_final_df['yield_kg_per_ha'].isna().sum())

print("\nYield availability by crop:")
yield_check = (
    fao_final_df
    .groupby('item')
    .agg(
        total_rows=('year', 'count'),
        yield_rows=('yield_kg_per_ha', 'count'),
        min_year=('year', 'min'),
        max_year=('year', 'max')
    )
    .sort_values('yield_rows', ascending=False)
)

print(yield_check)

Final dataset shape: (252, 36)

Missing yield values:
0

Yield availability by crop:
                            total_rows  yield_rows  min_year  max_year
item                                                                  
Bananas                             14          14      2010      2023
Barley and products                 14          14      2010      2023
Cassava and products                14          14      2010      2023
Coconuts - Incl Copra               14          14      2010      2023
Coffee and products                 14          14      2010      2023
Groundnuts                          14          14      2010      2023
Lemons, Limes and products          14          14      2010      2023
Maize and products                  14          14      2010      2023
Millet and products                 14          14      2010      2023
Onions                              14          14      2010      2023
Oranges, Mandarines                 14          14      2010   

## Check missing values

In [55]:
# ============================================================
# CHECK TOTAL VS EXPECTED COVERAGE
# ============================================================

total_rows = len(fao_final_df)
yield_rows = fao_final_df['yield_kg_per_ha'].notna().sum()

print("Total rows:", total_rows)
print("Rows with yield:", yield_rows)
print("Missing rows:", total_rows - yield_rows)

Total rows: 252
Rows with yield: 252
Missing rows: 0


## Creating lags for our data

- Lag features (yield lags and production lags) will help capture trends over time, seasonality of crops and also momentum effect

In [56]:
fao_final_df = fao_final_df.sort_values(['area', 'item', 'year']).copy()

In [57]:
# Yield lags
fao_final_df['yield_lag_1'] = fao_final_df.groupby(['area', 'item'])['yield_kg_per_ha'].shift(1)
fao_final_df['yield_lag_2'] = fao_final_df.groupby(['area', 'item'])['yield_kg_per_ha'].shift(2)

# Production lags
fao_final_df['production_lag_1'] = fao_final_df.groupby(['area', 'item'])['production'].shift(1)
fao_final_df['production_lag_2'] = fao_final_df.groupby(['area', 'item'])['production'].shift(2)

In [58]:
print(fao_final_df[
    ['yield_lag_1','yield_lag_2','production_lag_1','production_lag_2']
].isnull().sum())

yield_lag_1         18
yield_lag_2         36
production_lag_1    18
production_lag_2    36
dtype: int64


In [59]:
fao_final_df['yield_kg_per_ha'].unique()

array([12500. , 12400. , 12000. , 11923.1, 11987.3, 12206.7, 12230. ,
       12197.1, 12166.7, 12143.9, 12126.6, 12145.6, 12445.7, 12239.1,
        2556.2,  3464.1,  4394.9,  3348.1,  3037.4,  3679.3,  3591.7,
        3666.7,  3913.4,  4433.1,  3162.9,  2486.4,  2672.9,  2473.4,
        5252.1, 11230.9, 12727.3, 14247. , 13471.3, 14098.7, 12288.6,
       11541.3, 15359. , 14176.5, 14358.9, 11632. , 15049.8, 15626.5,
        1488. ,  1727. ,  2850.8,  1636.5,  1648.4,  2288.8,  2430.9,
        1137.4,  1270.6,  1295.5,  1295.7,  1115.9,   992.2,  1103.7,
         262.5,   314. ,   446.3,   362.5,   450. ,   370. ,   404.4,
         336.7,   358. ,   372.1,   308.3,   318.9,   474.4,   435.2,
         564.7,   948.1,  1503.6,  5434.2,  2598.2,  2575.6,  1866.8,
        1553.4,  1156.3,   968.6,  1000.4,   903.5,  1294.9,  1147. ,
       12812.4,  9701.6,  9355.4, 14366.3,  9553.6,  9750.7, 10980.4,
        8069.7,  7801.4,  8547.6, 21306.7, 12523.3,  8059.3, 11257.1,
        1725.1,  158

In [60]:
fao_final_df.describe()

,year,domestic_supply_quantity,export_quantity,feed,food,import_quantity,losses,other_uses_non_food,processing,production,...,gdp_usd_2015,agriculture_value_added_usd_2015,real_price_lcu_per_tonne,gdp_kes_2015_equiv,agriculture_value_added_kes_2015_equiv,yield_kg_per_ha,yield_lag_1,yield_lag_2,production_lag_1,production_lag_2
count,252.000000,252.000000,251.000000,113.000000,252.000000,244.000000,252.000000,76.000000,117.000000,252.000000,...,252.000000,252.000000,252.000000,2.520000e+02,2.520000e+02,252.000000,234.000000,216.000000,234.000000,216.000000
mean,2016.500000,796.166667,20.290837,99.132743,685.936508,217.352459,41.011905,23.618421,22.931624,605.035714,...,75643.196763,13942.124225,59829.463319,7.790066e+06,1.419150e+06,8610.179762,8601.708547,8634.012963,594.145299,593.462963
std,4.039151,1133.913818,41.328139,226.362679,945.378621,510.512087,54.386069,35.517495,23.962516,931.777487,...,13405.618608,1339.672689,93395.407582,2.562197e+06,3.459431e+05,8247.853172,8243.392285,8262.029106,909.996338,915.394488
min,2010.000000,-43.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,...,55768.208320,11821.199977,7323.577526,4.418691e+06,9.366309e+05,262.500000,262.500000,262.500000,11.000000,11.000000
25%,2013.000000,71.750000,0.000000,0.000000,54.500000,1.000000,4.000000,2.000000,0.000000,72.750000,...,63571.883415,12900.631019,25383.011881,5.474994e+06,1.111039e+06,1518.750000,1524.650000,1546.850000,72.250000,73.000000
50%,2016.500000,180.000000,2.000000,9.000000,146.500000,7.000000,15.000000,4.000000,18.000000,162.000000,...,74494.823721,13752.632144,35653.903577,7.633859e+06,1.408971e+06,6411.400000,6411.400000,6411.400000,159.000000,154.500000
75%,2020.000000,1090.250000,18.000000,40.000000,1072.500000,38.250000,53.500000,42.000000,39.000000,710.500000,...,84294.124217,15239.429327,53033.731122,8.948700e+06,1.652212e+06,12704.050000,12719.550000,12736.525000,711.500000,716.500000
max,2023.000000,5221.000000,227.000000,1247.000000,4035.000000,2557.000000,245.000000,171.000000,84.000000,4285.000000,...,100109.927037,16224.299115,848699.462957,1.400001e+07,2.268910e+06,35813.100000,35813.100000,35813.100000,4014.000000,4014.000000


## YIELD PREDICTION

## Model 1: Random Forest Regresser Model

In predicting the yield, we used the following key features:

- fertilizers (N,P,K) - This directly affect the growth of any crop

- pesticides - basically crop protection chemicals that help reduce losses from pests, weeds and disease

- Fertilizers and pesticides are the strongest biological drivers of yield.

- Economic indicators(GDP,CPI,exchange rate)

- Supply variables(production, imports,domestic suply quantity import quality)- will assist with market avalability and also food system pressure.

In [61]:
# Features
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne', 'yield_lag_1','yield_lag_2','production_lag_1','production_lag_2',
]

df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy()

# splitting our data using time test split
train = df_model[df_model['year'] <= 2020]
test = df_model[df_model['year'] > 2020]


# Features & target
X_train = train[features]
y_train = train['yield_kg_per_ha']

X_test = test[features]
y_test = test['yield_kg_per_ha']


# handling the missing values from the latest merging
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Model
model_yield = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# fitting the model
model_yield.fit(X_train, y_train)

# predicting our model
y_pred = model_yield.predict(X_test)


print("Random Forest Yield Prediction (Time Split)")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² :", r2_score(y_test, y_pred))

Random Forest Yield Prediction (Time Split)
MAE: 1702.6824629629614
R² : 0.8979966169334802


#### Random Forest Regressor Model result interpratation
From our statistics, the **mean yield_per_kg_ha is 18,081.86kgs**.

- From the MAE, our model **predicition is off by 2097.36kgs with an error of 11.59%**.

- The **R² : 0.8640598109608766**, indicates that the **model can explain 86%** of the variation variation in yield.


# Random Forest model

In [62]:
# ============================================================
# RANDOM FOREST YIELD MODEL PER CROP
# ============================================================
# This trains a separate Random Forest model for each crop.
# We use a time-based split:
# - train: 2010–2020
# - test: 2021–2023
#
# We evaluate using MAE and RMSE only.

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

# Features
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne',
    'yield_lag_1', 'yield_lag_2',
    'production_lag_1', 'production_lag_2'
]

rf_results = []

for crop in fao_final_df['item'].unique():
    
    # Filter one crop
    df_crop = fao_final_df[fao_final_df['item'] == crop].copy()
    df_crop = df_crop.dropna(subset=['yield_kg_per_ha'])
    df_crop = df_crop.sort_values('year')
    
    # Skip crops with too little data
    if len(df_crop) < 8:
        continue
    
    # Time-based split
    train = df_crop[df_crop['year'] <= 2020]
    test = df_crop[df_crop['year'] > 2020]
    
    if len(test) == 0:
        continue
    
    # Features and target
    X_train = train[features]
    y_train = train['yield_kg_per_ha']
    
    X_test = test[features]
    y_test = test['yield_kg_per_ha']
    
    # Handle missing values
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    
    # Train model
    model_yield = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    
    model_yield.fit(X_train, y_train)
    
    # Predict
    y_pred = model_yield.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Save results
    rf_results.append({
        'crop': crop,
        'n_obs': len(df_crop),
        'mae': mae,
        'rmse': rmse
    })

# Convert results to table
rf_results_df = pd.DataFrame(rf_results).sort_values('mae')

print("===== RANDOM FOREST PER CROP RESULTS =====")
print(rf_results_df)

print("\n===== RANDOM FOREST OVERALL PERFORMANCE =====")
print("Average MAE:", rf_results_df['mae'].mean())
print("Average RMSE:", rf_results_df['rmse'].mean())

===== RANDOM FOREST PER CROP RESULTS =====
                          crop  n_obs          mae         rmse
4          Coffee and products     14    41.856833    44.985629
7           Maize and products     14    69.467667    92.563576
0                      Bananas     14   114.104500   156.240564
8          Millet and products     14   133.684167   145.661135
14        Sorghum and products     14   218.287000   263.915626
3        Coconuts - Incl Copra     14   299.183333   307.155397
5                   Groundnuts     14   664.555667   680.800036
17          Wheat and products     14   722.782333   777.646321
15              Sweet potatoes     14   741.216833   760.022941
13           Rice and products     14   830.289833   868.670004
1          Barley and products     14   981.480500   985.459350
9                       Onions     14  1194.302833  1313.621751
12       Potatoes and products     14  1399.753167  1642.931603
10         Oranges, Mandarines     14  1405.121500  1650.9053

In [63]:
"""
The Random Forest model achieved strong predictive performance (MAE ~1,177 kg/ha; RMSE ~1,308 kg/ha), highlighting the importance of input and macro-level drivers in explaining yield variation. However, as it does not explicitly capture temporal dynamics, time-series models such as ARIMA and Prophet were introduced to model yield trends over time.
"""

'\nThe Random Forest model achieved strong predictive performance (MAE ~1,177 kg/ha; RMSE ~1,308 kg/ha), highlighting the importance of input and macro-level drivers in explaining yield variation. However, as it does not explicitly capture temporal dynamics, time-series models such as ARIMA and Prophet were introduced to model yield trends over time.\n'

## Model 2: XGBoost Model

In [64]:
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne','yield_lag_1','yield_lag_2', 'production_lag_1','production_lag_2'
]

df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy()

# For our time series data, we will use a time-split
train = df_model[df_model['year'] <= 2020]
test = df_model[df_model['year'] > 2020]

# Defining our target and features
X_train2 = train[features]
y_train2 = train['yield_kg_per_ha']

X_test2 = test[features]
y_test2 = test['yield_kg_per_ha']

# Handdling the missing values that arose from merging the previous dataset with the yield dataset
imputer = SimpleImputer(strategy="median")

X_train2 = imputer.fit_transform(X_train)
X_test2 = imputer.transform(X_test)

# XGB
model_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# fiting the model
model_xgb.fit(X_train2, y_train2)

# Evaluation of the model
y_pred2 = model_xgb.predict(X_test2)


mae = mean_absolute_error(y_test2, y_pred2)
rsme = np.sqrt(mean_squared_error(y_test2, y_pred2))
r2 = r2_score(y_test2, y_pred2)

print("XGBoost Yield Prediction Model (Time Split)")
print(f"MAE: {mae:.2f}")
print(f"R² : {r2:.2f}")
print(f"RMSE: {rsme:.2f}")


ValueError: Found input variables with inconsistent numbers of samples: [54, 3]

### XGBoost Yield Prediction Model Intepretation

From our statistics, **the mean yield_per_kg_ha is 18081.86**. Our models predicitions are **off by 2,368kgs** from our MAE metric.

- The **R² : 0.8120046957187079**, indicated that the **model can explain 81%** of yield variation.
Our model has an error of 13.09%.

- This means **our model can explain 81.2% of the variation in yield**
- We did not use **Adjusted R² because yield is not particularly linear**

In [ ]:
# ============================================================
# XGBOOST YIELD MODEL PER CROP
# ============================================================
# This trains a separate XGBoost model for each crop.
# We use the same time split as Random Forest:
# - train: <= 2020
# - test:  > 2020
#
# We evaluate using MAE and RMSE only.

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Features
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne',
    'yield_lag_1', 'yield_lag_2',
    'production_lag_1', 'production_lag_2'
]

xgb_results = []

for crop in fao_final_df['item'].unique():
    
    # 1. Filter one crop
    df_crop = fao_final_df[fao_final_df['item'] == crop].copy()
    df_crop = df_crop.dropna(subset=['yield_kg_per_ha'])
    df_crop = df_crop.sort_values('year')
    
    if len(df_crop) < 8:
        continue
    
    # 2. Time-based split
    train = df_crop[df_crop['year'] <= 2020]
    test = df_crop[df_crop['year'] > 2020]
    
    if len(test) == 0:
        continue
    
    # 3. Features and target
    X_train = train[features]
    y_train = train['yield_kg_per_ha']
    
    X_test = test[features]
    y_test = test['yield_kg_per_ha']
    
    # 4. Handle missing values
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    
    # 5. Train XGBoost
    model_xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    model_xgb.fit(X_train, y_train)
    
    # 6. Predict and evaluate
    y_pred = model_xgb.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    xgb_results.append({
        'crop': crop,
        'n_obs': len(df_crop),
        'mae': mae,
        'rmse': rmse
    })

# Results table
xgb_results_df = pd.DataFrame(xgb_results).sort_values('mae').reset_index(drop=True)

print("===== XGBOOST PER CROP RESULTS =====")
print(xgb_results_df)

print("\n===== XGBOOST OVERALL PERFORMANCE =====")
print("Average MAE:", xgb_results_df['mae'].mean())
print("Average RMSE:", xgb_results_df['rmse'].mean())

===== XGBOOST PER CROP RESULTS =====
                          crop  n_obs          mae         rmse
0          Coffee and products     14    30.510616    33.192381
1          Millet and products     14    47.660323    51.872476
2           Maize and products     14    48.935335    66.344515
3                      Bananas     14    79.752539   127.546503
4         Sorghum and products     14    90.132800    90.882241
5                   Groundnuts     14   286.918547   317.557368
6        Coconuts - Incl Copra     14   290.590104   305.829925
7               Sweet potatoes     14   292.993229   299.243673
8          Barley and products     14   795.024967   806.738465
9            Rice and products     14   991.662012  1202.628292
10          Wheat and products     14  1139.087988  1194.215636
11       Potatoes and products     14  1157.354818  1452.316504
12                      Onions     14  1205.287760  1320.003008
13         Oranges, Mandarines     14  1418.338802  1695.979263
14 

In [ ]:
"""The XGBoost model achieved strong predictive performance (MAE ~1,117 kg/ha; RMSE ~1,308 kg/ha), slightly outperforming Random Forest. This suggests that the additional regularisation and tree boosting in XGBoost may better capture complex relationships in the data. However, like Random Forest, XGBoost does not explicitly model temporal dynamics, which is why we also explore time-series models such as ARIMA and Prophet.
"""

## PRICE PREDICTION

In [ ]:
kamis_df = pd.read_csv('Data\kamis_data.csv',index_col=False)
kamis_df.drop(columns=['Unnamed: 0'], inplace=True)
kamis_df.head()

,Commodity,Market,Wholesale,Retail,Supply Volume,County,Date,commodity,commodity_id
0,Dry Maize,Ngurubani Market,50.0,60.00,5800.0,Kirinyaga,2026-04-17,Dry Maize,1
1,Dry Maize,Kerugoya,55.0,80.00,4600.0,Kirinyaga,2026-04-17,Dry Maize,1
2,Dry Maize,Voi Retail,0.0,65.00,360.0,Taita-Taveta,2026-04-17,Dry Maize,1
3,Dry Maize,Kawangware,55.0,65.00,0.0,Nairobi,2026-04-17,Dry Maize,1
4,Dry Maize,Mumias,0.0,61.36,0.0,Kakamega,2026-04-17,Dry Maize,1


In [ ]:
kamis_df.columns

Index(['Commodity', 'Market', 'Wholesale', 'Retail', 'Supply Volume', 'County',
       'Date', 'commodity', 'commodity_id'],
      dtype='str')

In [ ]:
kamis_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 432628 entries, 0 to 432627
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Commodity      432628 non-null  str    
 1   Market         432628 non-null  str    
 2   Wholesale      432628 non-null  float64
 3   Retail         432628 non-null  float64
 4   Supply Volume  432628 non-null  float64
 5   County         432628 non-null  str    
 6   Date           432628 non-null  str    
 7   commodity      432628 non-null  str    
 8   commodity_id   432628 non-null  int64  
dtypes: float64(3), int64(1), str(5)
memory usage: 29.7 MB


In [ ]:
# fix column names
kamis_df.columns = kamis_df.columns.str.lower().str.replace(" ", "_")

# convert date
kamis_df['date'] = pd.to_datetime(kamis_df['date'])

# extract time features
kamis_df['year'] = kamis_df['date'].dt.year
kamis_df['month'] = kamis_df['date'].dt.month

print(kamis_df.head())

   commodity            market  wholesale  retail  supply_volume  \
0  Dry Maize  Ngurubani Market       50.0   60.00         5800.0   
1  Dry Maize          Kerugoya       55.0   80.00         4600.0   
2  Dry Maize        Voi Retail        0.0   65.00          360.0   
3  Dry Maize        Kawangware       55.0   65.00            0.0   
4  Dry Maize            Mumias        0.0   61.36            0.0   

         county       date  commodity  commodity_id  year  month  
0     Kirinyaga 2026-04-17  Dry Maize             1  2026      4  
1     Kirinyaga 2026-04-17  Dry Maize             1  2026      4  
2  Taita-Taveta 2026-04-17  Dry Maize             1  2026      4  
3       Nairobi 2026-04-17  Dry Maize             1  2026      4  
4      Kakamega 2026-04-17  Dry Maize             1  2026      4  


In [ ]:
kamis_df.columns = kamis_df.columns.str.lower()

In [ ]:
# standardise column names
kamis_df.columns = kamis_df.columns.str.lower().str.replace(" ", "_")

# remove duplicate columns
kamis_df = kamis_df.loc[:, ~kamis_df.columns.duplicated()]

print(kamis_df.columns.tolist())

['commodity', 'market', 'wholesale', 'retail', 'supply_volume', 'county', 'date', 'commodity_id', 'year', 'month']


In [ ]:
kamis_df = kamis_df.sort_values(['commodity', 'date'])

### Creating lags

In [ ]:
kamis_df = kamis_df.sort_values(['commodity', 'date'])


# PRICE LAGS

kamis_df['retail_lag_1'] = kamis_df.groupby('commodity')['retail'].shift(1)
kamis_df['retail_lag_7'] = kamis_df.groupby('commodity')['retail'].shift(7)

kamis_df['wholesale_lag_1'] = kamis_df.groupby('commodity')['wholesale'].shift(1)
kamis_df['wholesale_lag_7'] = kamis_df.groupby('commodity')['wholesale'].shift(7)

# ROLLING FEATURES
kamis_df['retail_rolling_7'] = (
    kamis_df.groupby('commodity')['retail']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

kamis_df['wholesale_rolling_7'] = (
    kamis_df.groupby('commodity')['wholesale']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

# PRICE SPREAD
kamis_df['price_spread'] = kamis_df['retail'] - kamis_df['wholesale']

# VOLATILITY
kamis_df['price_volatility_7'] = (
    kamis_df.groupby('commodity')['retail']
      .rolling(7)
      .std()
      .reset_index(0, drop=True)
)

# PRICE CHANGE
kamis_df['price_change_1'] = kamis_df.groupby('commodity')['retail'].diff(1)


# SUPPLY FEATURES
kamis_df['supply_lag_1'] = kamis_df.groupby('commodity')['supply_volume'].shift(1)

kamis_df['supply_rolling_7'] = (
    kamis_df.groupby('commodity')['supply_volume']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

kamis_df['supply_change_1'] = kamis_df.groupby('commodity')['supply_volume'].diff(1)

# RELATIVE PRICE LEVEL
kamis_df['price_vs_avg_7'] = kamis_df['retail'] / kamis_df['retail_rolling_7']


# CLEAN
kamis_df = kamis_df.dropna()

print(kamis_df.shape)
print(kamis_df.head())

(402367, 23)
                     commodity market  wholesale  retail  supply_volume  \
101512  African butter catfish   Aram        0.0   250.0           50.0   
101511  African butter catfish   Aram        0.0   250.0           32.0   
101510  African butter catfish   Aram        0.0   250.0           40.0   
101509  African butter catfish   Aram        0.0   250.0           40.0   
101508  African butter catfish   Aram        0.0   250.0           38.5   

       county       date  commodity_id  year  month  ...  wholesale_lag_7  \
101512  Siaya 2021-08-02            87  2021      8  ...              0.0   
101511  Siaya 2021-08-05            87  2021      8  ...            180.0   
101510  Siaya 2021-08-09            87  2021      8  ...              0.0   
101509  Siaya 2021-08-12            87  2021      8  ...              0.0   
101508  Siaya 2021-08-16            87  2021      8  ...              0.0   

        retail_rolling_7  wholesale_rolling_7  price_spread  \
101512    

## Modelling

In [ ]:
# Roadmap
1. Random Forest per commodity  
2. XGBoost per commodity  
3. ARIMA per commodity  
4. SARIMA per commodity  
5. Prophet per commodity  
6. Stacking

### XGBoost model

### Creating features

In [ ]:
features = [
    'retail_lag_1',
    'retail_lag_7',
    'wholesale_lag_1',
    'wholesale_lag_7',
    'retail_rolling_7',
    'wholesale_rolling_7',
    'price_spread',
    'price_volatility_7',
    'price_change_1',
    'supply_volume',
    'supply_lag_1',
    'supply_rolling_7',
    'supply_change_1',
    'price_vs_avg_7',
    'month',
    'commodity_id'
]

X = kamis_df[features]
y = kamis_df['retail']

print(X.shape)
print(X.head())


(402367, 16)
        retail_lag_1  retail_lag_7  wholesale_lag_1  wholesale_lag_7  \
101512         250.0         250.0              0.0              0.0   
101511         250.0         250.0              0.0            180.0   
101510         250.0         250.0              0.0              0.0   
101509         250.0         250.0              0.0              0.0   
101508         250.0         250.0              0.0              0.0   

        retail_rolling_7  wholesale_rolling_7  price_spread  \
101512             250.0            25.714286         250.0   
101511             250.0             0.000000         250.0   
101510             250.0             0.000000         250.0   
101509             250.0             0.000000         250.0   
101508             250.0             0.000000         250.0   

        price_volatility_7  price_change_1  supply_volume  supply_lag_1  \
101512                 0.0             0.0           50.0          32.0   
101511                 0.

### Train_test_split and modelling

In [ ]:
train = kamis_df[kamis_df['date'] <= '2023-12-31']
test = kamis_df[kamis_df['date'] > '2023-12-31']

X_train = train[features]
y_train = train['retail']

X_test = test[features]
y_test = test['retail']

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# Modelling
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=100,
    max_depth=6,        # slightly deeper for big data
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1           # use all cores
)


#fitting the model
model.fit(X_train, y_train)

#predicting the model
y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("Test RMSE:", rmse)
print("MAE:",  mae)

Train size: (228711, 16)
Test size: (173656, 16)
Test RMSE: 782.7154078554267
MAE: 9.749693640345637


For XGBoost, R2 and adjusted R2 are not very meaningful because:

they assumes linear relationships and XGBoost is non-linear

### Train separate models per commodity

In [ ]:
id="fix_per_commodity"

all_preds = []
all_actuals = []

for commodity in kamis_df['commodity'].unique():
    train_c = train[train['commodity'] == commodity]
    test_c = test[test['commodity'] == commodity]

    if len(test_c) < 10:
        continue

    X_train_c = train_c[features]
    y_train_c = train_c['retail']

    X_test_c = test_c[features]
    y_test_c = test_c['retail']

    model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        random_state=42
    )


    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)

    all_preds.extend(preds)
    all_actuals.extend(y_test_c)

# Evaluate globally
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("Fixed RMSE:", rmse)
print("Fixed MAE:", mae)

Fixed RMSE: 209.6913661077517
Fixed MAE: 11.987146249930205


In [ ]:
"""
The XGBoost model demonstrates strong predictive performance (MAE ~11.99 KES; RMSE ~209.69 KES), indicating that it is able to capture complex, non-linear relationships in the data. By incorporating supply, lagged prices, and market dynamics, the model improves prediction accuracy compared to traditional time-series approaches, although some larger errors remain
"""

## Random Forest model

In [ ]:
# ============================================================
# RANDOM FOREST PRICE MODEL
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# -----------------------------
# TRAIN / TEST SPLIT (TIME-BASED)
# -----------------------------
train = kamis_df[kamis_df['date'] <= '2023-12-31']
test = kamis_df[kamis_df['date'] > '2023-12-31']

# -----------------------------
# FEATURES & TARGET
# -----------------------------
X_train = train[features]
y_train = train['retail']

X_test = test[features]
y_test = test['retail']

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# -----------------------------
# HANDLE MISSING VALUES
# -----------------------------
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# -----------------------------
# MODEL
# -----------------------------
model_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# -----------------------------
# TRAIN
# -----------------------------
model_rf.fit(X_train, y_train)

# -----------------------------
# PREDICT
# -----------------------------
y_pred = model_rf.predict(X_test)

# -----------------------------
# EVALUATE
# -----------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nRandom Forest Results")
print("MAE:", mae)
print("RMSE:", rmse)

Train size: (228711, 16)
Test size: (173656, 16)

Random Forest Results
MAE: 7.043731587314536
RMSE: 50.20862340191269


In [ ]:
"""
The Random Forest model achieves the best performance among all models (MAE ~7.04 KES; RMSE ~50.21 KES), indicating highly stable and accurate predictions across commodities. The model benefits from averaging across multiple decision trees, which reduces variance and improves robustness, particularly in handling noisy and heterogeneous price data.
"""


## ARIMA model

In [ ]:
# ============================================================
# ARIMA MODEL (PER COMMODITY — TIME SERIES BASELINE)
# ============================================================
# ARIMA models price using past values only.
#
# It captures:
# - trend (overall direction)
# - momentum (recent changes)
#
# It does NOT use external features like supply or volatility.

import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

all_preds = []
all_actuals = []

for commodity in kamis_df['commodity'].unique():
    
    # -----------------------------
    # 1. PREPARE DATA
    # -----------------------------
    df_com = kamis_df[kamis_df['commodity'] == commodity].copy()
    df_com = df_com.sort_values('date')
    
    # -----------------------------
    # 2. TIME SPLIT
    # -----------------------------
    train_c = df_com[df_com['date'] <= '2023-12-31']
    test_c = df_com[df_com['date'] > '2023-12-31']
    
    if len(test_c) < 10:
        continue
    
    try:
        # -----------------------------
        # 3. TRAIN ARIMA MODEL
        # -----------------------------
        model = ARIMA(train_c['retail'], order=(1,1,1))
        model_fit = model.fit()
        
        # -----------------------------
        # 4. FORECAST
        # -----------------------------
        preds = model_fit.forecast(steps=len(test_c))
        
        all_preds.extend(preds)
        all_actuals.extend(test_c['retail'].values)
    
    except:
        continue

# -----------------------------
# 5. EVALUATION
# -----------------------------
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("ARIMA Results")
print("RMSE:", rmse)
print("MAE:", mae)

ARIMA Results
RMSE: 264.8656241266683
MAE: 53.03689785395104


In [ ]:
"""The ARIMA model performs significantly worse (MAE ~53.04 KES; RMSE ~264.87 KES), reflecting its limitation in relying only on past price values. Without incorporating additional drivers such as supply or volatility, it struggles to capture the complexity of price movements.
"""

## SARIMA model

In [ ]:
# ============================================================
# SARIMA MODEL (PER COMMODITY — SEASONAL TIME SERIES)
# ============================================================
# SARIMA extends ARIMA by adding seasonality.
#
# It captures:
# - trend
# - momentum
# - repeating seasonal patterns
#
# Here, we test weekly seasonality using seasonal_order=(1,1,1,7).

import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

all_preds = []
all_actuals = []

for commodity in kamis_df['commodity'].unique():
    
    # 1. PREPARE DATA
    df_com = kamis_df[kamis_df['commodity'] == commodity].copy()
    df_com = df_com.sort_values('date')
    
    # 2. TIME SPLIT
    train_c = df_com[df_com['date'] <= '2023-12-31']
    test_c = df_com[df_com['date'] > '2023-12-31']
    
    if len(test_c) < 10:
        continue
    
    try:
        # 3. TRAIN SARIMA MODEL
        model = SARIMAX(
            train_c['retail'],
            order=(1,1,1),
            seasonal_order=(1,1,1,7),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        
        model_fit = model.fit(disp=False)
        
        # 4. FORECAST
        preds = model_fit.forecast(steps=len(test_c))
        
        all_preds.extend(preds)
        all_actuals.extend(test_c['retail'].values)
    
    except:
        continue

# 5. EVALUATION
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("SARIMA Results")
print("RMSE:", rmse)
print("MAE:", mae)

SARIMA Results
RMSE: 18048.76930903851
MAE: 242.11695552448415


In [ ]:
"""The SARIMA model performs poorly (MAE ~242.12 KES; RMSE ~18,048.77 KES), indicating large prediction errors. This suggests that seasonal time-series models are not well suited for this dataset, likely due to high variability, differing price scales across commodities, and the absence of consistent seasonal patterns.
"""

## Prophet model

In [ ]:
# ============================================================
# PROPHET MODEL (PER COMMODITY — PRICE TIME SERIES)
# ============================================================
# Prophet is a time-series model.
#
# It uses:
# - past retail prices
# - trend changes
# - weekly seasonality
#
# It does NOT use engineered features such as supply_volume,
# price_spread, or rolling averages.
#
# We run Prophet separately for each commodity, then combine all
# predictions and actual values to calculate overall MAE and RMSE.

import warnings
warnings.filterwarnings("ignore")


import logging
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

all_preds = []
all_actuals = []

for commodity in kamis_df['commodity'].unique():
    
    # 1. Prepare one commodity time series
    df_com = kamis_df[kamis_df['commodity'] == commodity].copy()
    df_com = df_com.sort_values('date')
    
    # 2. Time-based split
    train_c = df_com[df_com['date'] <= '2023-12-31']
    test_c = df_com[df_com['date'] > '2023-12-31']
    
    if len(test_c) < 10:
        continue
    
    try:
        # 3. Convert data into Prophet format
        # Prophet requires:
        # ds = date column
        # y  = target column
        prophet_train = train_c[['date', 'retail']].rename(
            columns={'date': 'ds', 'retail': 'y'}
        )
        
        # 4. Train Prophet model
        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=True,
            yearly_seasonality=False
        )
        
        model.fit(prophet_train)
        
        # 5. Prepare test dates for prediction
        future = test_c[['date']].rename(columns={'date': 'ds'})
        
        # 6. Predict retail prices
        forecast = model.predict(future)
        preds = forecast['yhat'].values
        
        # 7. Store predictions and actual values
        all_preds.extend(preds)
        all_actuals.extend(test_c['retail'].values)
    
    except:
        continue

# 8. Evaluate Prophet model
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("Prophet Results")
print("RMSE:", rmse)
print("MAE:", mae)

Prophet Results
RMSE: 1622.2056533726882
MAE: 326.6518209923209


In [ ]:
"""
The Prophet model performs moderately but is weaker than the machine learning models (MAE ~326.65 KES; RMSE ~1,622.21 KES), indicating relatively large prediction errors. While Prophet is able to capture trend and weekly seasonality, it struggles with the high variability and cross-commodity differences in the data. This suggests that, similar to ARIMA and SARIMA, models that rely only on past price patterns are less effective than those that incorporate additional features such as supply and lagged dynamics.
"""

Model            | MAE (KES) | RMSE (KES) | Performance Summary
---------------------------------------------------------------------------
Random Forest    | 7.04      | 50.21      | Best – most accurate and stable
XGBoost          | 11.99     | 209.69     | Strong – captures complex patterns
ARIMA            | 53.04     | 264.87     | Weak – relies only on past prices
Prophet          | 326.65    | 1622.21    | Poor – struggles with variability
SARIMA           | 242.12    | 18048.77   | Very poor – unstable predictions, should not be stacked

## Stacking

In [ ]:
# ============================================================
# STACKED MODEL (RF + XGBOOST + ARIMA)
# ============================================================
# This combines predictions from:
# - Random Forest
# - XGBoost
# - ARIMA
#
# We skip commodities that do not have enough train or test data.

import warnings
warnings.filterwarnings("ignore")

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
import numpy as np

all_preds = []
all_actuals = []
skipped = []

for commodity in kamis_df['commodity'].unique():
    
    # 1. Filter one commodity
    df_com = kamis_df[kamis_df['commodity'] == commodity].copy()
    df_com = df_com.sort_values('date')
    
    # 2. Time split
    train_c = df_com[df_com['date'] <= '2023-12-31']
    test_c = df_com[df_com['date'] > '2023-12-31']
    
    # 3. Skip commodities without enough train/test data
    if len(train_c) < 20 or len(test_c) < 10:
        skipped.append({
            'commodity': commodity,
            'train_rows': len(train_c),
            'test_rows': len(test_c)
        })
        continue
    
    try:
        # 4. Define features and target
        X_train = train_c[features]
        y_train = train_c['retail']
        
        X_test = test_c[features]
        y_test = test_c['retail']
        
        # 5. XGBoost model
        xgb = XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1
        )
        xgb.fit(X_train, y_train)
        pred_xgb = xgb.predict(X_test)
        
        # 6. Random Forest model
        rf = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        pred_rf = rf.predict(X_test)
        
        # 7. ARIMA model
        arima = ARIMA(train_c['retail'], order=(1, 1, 1)).fit()
        pred_arima = np.array(arima.forecast(steps=len(test_c)))
        
        # 8. Stack predictions
        stack_X = np.column_stack([
            pred_rf,
            pred_xgb,
            pred_arima
        ])
        
        # 9. Meta-model
        meta_model = LinearRegression()
        meta_model.fit(stack_X, y_test)
        
        final_pred = meta_model.predict(stack_X)
        
        all_preds.extend(final_pred)
        all_actuals.extend(y_test)
    
    except Exception as e:
        skipped.append({
            'commodity': commodity,
            'train_rows': len(train_c),
            'test_rows': len(test_c),
            'error': str(e)
        })
        continue

# 10. Evaluation
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("STACKED MODEL RESULTS")
print("RMSE:", rmse)
print("MAE:", mae)

print("\nSkipped commodities:", len(skipped))

STACKED MODEL RESULTS
RMSE: 152.82851572953672
MAE: 20.34823079253102

Skipped commodities: 31


In [ ]:
"""
The stacked model combines Random Forest, XGBoost, and ARIMA into one final prediction. It achieves moderate performance (MAE ~20.35 KES; RMSE ~152.83 KES), showing that it balances the strengths of different models. While it is not as accurate as Random Forest alone, it provides a more general approach by combining multiple methods.
"""

## ALTENATIVE MODELS

## ARIMA model(Baseline Time Series)

- Best for non-seasonal data
- Simple trends

- ARIMA can be used to forecast both yield or price.

In [ ]:
# Prepare the data
df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy()

# Convert year to datetime
df_model['year'] = pd.to_datetime(df_model['year'], format='%Y')

# Sort and set index
df_model = df_model.sort_values('year') # for time series data, year has to be soted or there will be wrong learning
df_model.set_index('year', inplace=True)  # using  year as the index

# Target variable
y = df_model['yield_kg_per_ha']


# Train-test split (time-based)
train = y.iloc[:-5]
test = y.iloc[-5:]

# Fit ARIMA model
# -----------------------------
model = ARIMA(train, order=(1,1,1))
model_fit = model.fit()

# Forecast
forecast = model_fit.forecast(steps=len(test))

# Align index with test
forecast.index = test.index

# Evaluation our model
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("ARIMA Yield Forecast")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))

ARIMA Yield Forecast
MAE : 5953.03
RMSE: 6400.51


c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction r

The ARIMA model produced a Mean Absolute Error (MAE) of 5953.03 and a Root Mean Squared Error (RMSE) of 6400.51, indicating relatively high prediction errors. In contrast, the Random Forest Regressor achieved a significantly lower MAE of 2097.36 and an R² score of 0.864, explaining approximately 86.4% of the variance in yield.

This suggests that the **Random Forest model substantially outperforms ARIMA for yield prediction**. The primary reason is that ARIMA relies solely on historical yield values, whereas **Random Forest leverages multiple explanatory variables such as fertilizer use, economic indicators, and production factors**.

Therefore, yield in this dataset is better modeled as a function of multiple features rather than as a purely time-dependent process.


This clearly shows that **yield prediction is feature driven and not just time-driven**. Yield also depends on other factors such as **weather, input, economy** etc

## Model 2: ARIMAX(ARIMA + OUR FEATURES)
- Uses features as external regressors.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer


# Definng our features
# fetures are the external drivers
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne',
    'yield_lag_1','yield_lag_2',
    'production_lag_1','production_lag_2'
]

# Prepare our data
df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy() # keeps only rows where yield exists
df_model['year'] = pd.to_datetime(df_model['year'], format='%Y') # converts year into a proper time object
df_model = df_model.sort_values('year') # ensures data is in chronological order
df_model.set_index('year', inplace=True) # year becomes index(time axis)


# Train-test-Split
train = df_model[df_model.index.year <= 2020]   # splitting our data by time, uses past data to predict the future
test = df_model[df_model.index.year > 2020]

y_train = train['yield_kg_per_ha'] ## splitting the target(y)
y_test = test['yield_kg_per_ha']
# y_train-(past yield) what the model learns
#y_test-future yields(what you evaluate on)

X_train = train[features]
X_test = test[features]
# we are giving arimax X_train(past economic/agriculture drivers and X_test (future drivers) used for prediction)

## Arimax model learns: Yield = f(past yield + external factors)

###
# Handle missing values
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# ARIMAX model
model_arimax = ARIMA(y_train, exog=X_train, order=(1,1,1))
fit_arimax = model_arimax.fit()

# Forecast
pred_arimax = fit_arimax.forecast(steps=len(y_test), exog=X_test)

# Evaluation our model
mae_arimax = mean_absolute_error(y_test, pred_arimax)
rmse_arimax = np.sqrt(mean_squared_error(y_test, pred_arimax))

print("ARIMAX Results")
print("MAE :", round(mae_arimax, 2))
print("RMSE:", round(rmse_arimax, 2))

c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ARIMAX Results
MAE : 156898.23
RMSE: 177658.47


c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


## SARIMA model( Handles seasonality well)
**KAMIS data could probably be modelled using SARIMA**

- best for agricultural data
- harvest patterns
- if data is yearly, seasonality might break
if monthly, SARIMA becomes very powerful

However **SARIMAX only works if there is one crop, one region and long continous yearly history, otherwise quite unstable**

## Prophet Model
- Best for messy agricultural data
- Missing values
- Strong trends and seasonality
- Business-style forecasting

- For yield model, we could explore using **SARIMA** and add external features auch as fertlizer.

- For **price model** we could start by using **ARIMA and upgrade to Prophet**

In [ ]:
pip install prophet

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 730.2 kB/s eta 0:00:16
   - -------------------------------------- 0.5/12.1 MB 730.2 kB/s eta 0:00:16
   - -------------------------------------- 0.5/12.1 MB 730.2 kB/s eta 0:00:16
   - -------------------------------------- 0.5/12.1 MB 730.2 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/12.1 MB 653.7 kB/s eta 0:00:17
   ----- ---------------------------------- 1.6/12.1 MB 883.2 kB/s eta 0:00:12
   ------ --------------------------------- 2.1/12.1 MB 1.1 MB/s eta 0:00:10
   ------- -------------------------------- 2.4/12.1 MB 1.1 MB/s eta 0:00:09
   --------- ------------------------------ 2.9/12.1 MB 1.3 MB/s eta 0:00:08
   ----------- 

### Abigael's models

## Yield  model

In [67]:
# ============================================================
# PREPARE TIME SERIES DATA (YIELD)
# ============================================================

yield_ts = (
    fao_final_df[['year', 'yield_kg_per_ha']]
    .dropna()
    .drop_duplicates(subset=['year'])
    .sort_values('year')
)

In [66]:
# ============================================================
# TIME-BASED TRAIN / TEST SPLIT
# ============================================================
# For time-series models, we must simulate real forecasting.
# That means:
# - Train on past years
# - Test on future years
#
# We will:
# Train: 2010–2020
# Test:  2021–2023

train_ts = yield_ts[yield_ts['year'] <= 2020].copy()
test_ts = yield_ts[yield_ts['year'] > 2020].copy()

# Check the split
print("TRAIN DATA:")
print(train_ts)

print("\nTEST DATA:")
print(test_ts)

print("\nTrain shape:", train_ts.shape)
print("Test shape:", test_ts.shape)

"""
Correct principle for time-series forecasting:
- more past data = better model
-so we Train = as much past as possible and test with most recent years
-Train: 2010–2020 (learn history)  
Test:  2021–2023 (predict future)
"""

TRAIN DATA:
    year  yield_kg_per_ha
0   2010          12500.0
1   2011          12400.0
2   2012          12000.0
3   2013          11923.1
4   2014          11987.3
5   2015          12206.7
6   2016          12230.0
7   2017          12197.1
8   2018          12166.7
9   2019          12143.9
10  2020          12126.6

TEST DATA:
    year  yield_kg_per_ha
11  2021          12145.6
12  2022          12445.7
13  2023          12239.1

Train shape: (11, 2)
Test shape: (3, 2)


'\nCorrect principle for time-series forecasting:\n- more past data = better model\n-so we Train = as much past as possible and test with most recent years\n-Train: 2010–2020 (learn history)  \nTest:  2021–2023 (predict future)\n'

### ARIMA model for ALL crops seperately

In [ ]:
# ============================================================
# ARIMA MODEL FOR ALL CROPS
# ============================================================

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

results = []

# Loop through each crop
for crop in fao_final_df['item'].unique():
    
    # Prepare clean time series for each crop
    df_crop = (
        fao_final_df
        .loc[fao_final_df['item'] == crop, ['year', 'yield_kg_per_ha']]
        .dropna()
        .drop_duplicates(subset=['year'])
        .sort_values('year')
    )
    
    # Skip crops with too little data
    if len(df_crop) < 8:
        continue
    
    # Train-test split (time-based)
    train = df_crop[df_crop['year'] <= 2020]
    test = df_crop[df_crop['year'] > 2020]
    
    if len(test) == 0:
        continue
    
    try:
        # Fit ARIMA
        model = ARIMA(train['yield_kg_per_ha'], order=(1,1,1))
        model_fit = model.fit()
        
        # Forecast
        pred = model_fit.forecast(steps=len(test))
        
        y_true = test['yield_kg_per_ha'].values
        y_pred = np.array(pred)
        
        # Metrics
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        results.append({
            'crop': crop,
            'n_obs': len(df_crop),
            'mae': mae,
            'rmse': rmse
        })
        
    except Exception as e:
        # If model fails for a crop, skip it
        continue

# Convert to DataFrame
results_df = pd.DataFrame(results).sort_values('rmse')

# ============================================================
# OUTPUTS
# ============================================================

print("===== PER CROP RESULTS =====")
print(results_df)

print("\n===== OVERALL PERFORMANCE =====")
print("Average MAE:", results_df['mae'].mean())
print("Average RMSE:", results_df['rmse'].mean())

c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
 

===== PER CROP RESULTS =====
                          crop  n_obs          mae         rmse
4          Coffee and products     14    74.751203    82.751207
17          Wheat and products     14    88.551962    97.116741
7           Maize and products     14   138.614904   147.227044
0                      Bananas     14   181.855601   216.465805
5                   Groundnuts     14   316.380311   334.804146
3        Coconuts - Incl Copra     14   532.301120   538.800094
1          Barley and products     14   609.916268   616.705642
14        Sorghum and products     14   676.869271   697.367394
15              Sweet potatoes     14   674.107917   702.051694
8          Millet and products     14   793.608925   798.709811
10         Oranges, Mandarines     14   759.644896   876.227152
13           Rice and products     14  1004.737998  1229.918662
12       Potatoes and products     14  1364.311722  1619.265021
2         Cassava and products     14  1525.279149  1775.318799
11     Pine

c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


* On average, the model achieved an MAE of about 1,382 kg/ha and an RMSE of about 1,510 kg/ha, meaning that predictions are typically off by around 1.4–1.5 tonnes per hectare. 
* Given that most crop yields range between 20,000 and 30,000 kg/ha, this corresponds to an error of roughly 5–8 percent. 
* This indicates that ARIMA captures the general trend in yields reasonably well, but its performance varies significantly across crops, performing better for stable crops like coffee and wheat, and worse for more volatile crops such as tomatoes and onions.

## SARIMA model 

In [ ]:
# ============================================================
# SARIMA MODEL FOR ALL CROPS
# ============================================================
# SARIMA extends ARIMA by adding a seasonal component.
# Even though our data is yearly (weak seasonality),
# we test SARIMA to see if it improves performance.

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

sarima_results = []

for crop in fao_final_df['item'].unique():
    
    df_crop = (
        fao_final_df
        .loc[fao_final_df['item'] == crop, ['year', 'yield_kg_per_ha']]
        .dropna()
        .drop_duplicates(subset=['year'])
        .sort_values('year')
    )
    
    if len(df_crop) < 8:
        continue
    
    train = df_crop[df_crop['year'] <= 2020]
    test = df_crop[df_crop['year'] > 2020]
    
    if len(test) == 0:
        continue
    
    try:
        # SARIMA model (simple seasonal assumption)
        model = SARIMAX(
            train['yield_kg_per_ha'],
            order=(1,1,1),
            seasonal_order=(1,0,0,2),  # light seasonality
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        
        model_fit = model.fit(disp=False)
        
        pred = model_fit.forecast(steps=len(test))
        
        y_true = test['yield_kg_per_ha'].values
        y_pred = np.array(pred)
        
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        sarima_results.append({
            'crop': crop,
            'n_obs': len(df_crop),
            'mae': mae,
            'rmse': rmse
        })
        
    except:
        continue

sarima_df = pd.DataFrame(sarima_results).sort_values('rmse')

print("===== SARIMA PER CROP RESULTS =====")
print(sarima_df)

print("\n===== SARIMA OVERALL PERFORMANCE =====")
print("Average MAE:", sarima_df['mae'].mean())
print("Average RMSE:", sarima_df['rmse'].mean())

===== SARIMA PER CROP RESULTS =====
                          crop  n_obs           mae          rmse
4          Coffee and products     14     96.322795    108.323298
7           Maize and products     14    131.285754    138.866465
0                      Bananas     14    123.894342    173.740077
3        Coconuts - Incl Copra     14    141.894664    173.787077
5                   Groundnuts     14    173.236523    182.914202
17          Wheat and products     14    357.134397    424.759673
8          Millet and products     14    680.542166    692.671630
13           Rice and products     14    493.851073    782.193839
14        Sorghum and products     14    936.335355    945.758645
10         Oranges, Mandarines     14    893.982383   1032.409144
1          Barley and products     14   1086.637726   1116.593567
12       Potatoes and products     14   1428.811238   1664.661666
2         Cassava and products     14   1731.874659   1798.446002
11     Pineapples and products     14   

* The SARIMA model does not improve performance compared to ARIMA. In fact, it results in higher average errors, with MAE increasing from about 1,382 kg/ha to 1,726 kg/ha and RMSE from about 1,510 kg/ha to 1,907 kg/ha. 
* This suggests that adding a seasonal component does not enhance predictive performance for this dataset. 
* This is likely because the data is annual, and therefore does not exhibit strong seasonal patterns that SARIMA is designed to capture.
* we drop SARIMA  and not include it in final stacking

## Prophet model

In [ ]:
# ============================================================
# PROPHET MODEL FOR ALL CROPS (DEBUG VERSION)
# ============================================================
# Prophet is another time-series model.
# It is useful for capturing trends and changes over time.
#
# Prophet requires two columns:
# ds = date column
# y  = target variable
#
# We run Prophet separately for each crop because each crop has
# its own yield pattern over time.
#
# This debug version prints SUCCESS or FAILED for each crop so
# we can see exactly what is happening.

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

prophet_results = []

for crop in fao_final_df['item'].unique():
    
    # --------------------------------------------------------
    # 1. Create one clean time series for this crop
    # --------------------------------------------------------
    df_crop = (
        fao_final_df
        .loc[fao_final_df['item'] == crop, ['year', 'yield_kg_per_ha']]
        .dropna()
        .drop_duplicates(subset=['year'])
        .sort_values('year')
    )
    
    # Prophet needs enough historical data to learn a trend
    if len(df_crop) < 8:
        print(f"{crop} → SKIPPED: too few observations")
        continue
    
    # --------------------------------------------------------
    # 2. Convert data into Prophet format
    # --------------------------------------------------------
    df_prophet = df_crop.copy()
    
    # Convert year into a real date.
    # We use end-of-year dates because the data is annual.
    df_prophet['ds'] = pd.to_datetime(df_prophet['year'].astype(str) + '-12-31')
    
    # Prophet target column must be named 'y'
    df_prophet['y'] = df_prophet['yield_kg_per_ha']
    
    # --------------------------------------------------------
    # 3. Time-based train/test split
    # --------------------------------------------------------
    train = df_prophet[df_prophet['year'] <= 2020].copy()
    test = df_prophet[df_prophet['year'] > 2020].copy()
    
    if len(test) == 0:
        print(f"{crop} → SKIPPED: no test years")
        continue
    
    try:
        # ----------------------------------------------------
        # 4. Fit Prophet model
        # ----------------------------------------------------
        # We switch off daily/weekly/yearly seasonality because
        # this is annual data, not daily or monthly data.
        model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False
        )
        
        model.fit(train[['ds', 'y']])
        
        # ----------------------------------------------------
        # 5. Create future dates matching the test period
        # ----------------------------------------------------
        future = pd.DataFrame({
            'ds': test['ds'].values
        })
        
        forecast = model.predict(future)
        
        # ----------------------------------------------------
        # 6. Evaluate predictions
        # ----------------------------------------------------
        y_true = test['y'].values
        y_pred = forecast['yhat'].values
        
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        prophet_results.append({
            'crop': crop,
            'n_obs': len(df_crop),
            'mae': mae,
            'rmse': rmse
        })
        
        print(f"{crop} → SUCCESS")
        
    except Exception as e:
        print(f"{crop} → FAILED: {e}")

# ============================================================
# 7. Convert results into a table
# ============================================================

prophet_df = pd.DataFrame(prophet_results)

if len(prophet_df) > 0:
    prophet_df = prophet_df.sort_values('rmse')
    
    print("\n===== PROPHET PER CROP RESULTS =====")
    print(prophet_df)
    
    print("\n===== PROPHET OVERALL PERFORMANCE =====")
    print("Average MAE:", prophet_df['mae'].mean())
    print("Average RMSE:", prophet_df['rmse'].mean())
else:
    print("\nNo Prophet models were successfully fitted.")

06:29:08 - cmdstanpy - INFO - Chain [1] start processing
06:29:08 - cmdstanpy - INFO - Chain [1] done processing
06:29:08 - cmdstanpy - INFO - Chain [1] start processing


Bananas → SUCCESS


06:29:09 - cmdstanpy - INFO - Chain [1] done processing
06:29:09 - cmdstanpy - INFO - Chain [1] start processing


Barley and products → SUCCESS


06:29:09 - cmdstanpy - INFO - Chain [1] done processing
06:29:09 - cmdstanpy - INFO - Chain [1] start processing


...

Sweet potatoes → SUCCESS


06:29:15 - cmdstanpy - INFO - Chain [1] done processing
06:29:15 - cmdstanpy - INFO - Chain [1] start processing


Tomatoes and products → SUCCESS


06:29:16 - cmdstanpy - INFO - Chain [1] done processing


Wheat and products → SUCCESS

===== PROPHET PER CROP RESULTS =====
                          crop  n_obs          mae         rmse
4          Coffee and products     14    73.557271    77.411991
7           Maize and products     14   142.992070   145.861802
3        Coconuts - Incl Copra     14   177.756739   193.209525
0                      Bananas     14   200.350481   240.187838
5                   Groundnuts     14   257.717228   325.588141
8          Millet and products     14   396.321896   401.797737
14        Sorghum and products     14   479.064930   497.145726
13           Rice and products     14   641.463825   716.807273
17          Wheat and products     14   818.427915   828.913031
1          Barley and products     14  1448.154337  1452.043499
9                       Onions     14  1407.824948  1522.406630
16       Tomatoes and products     14  1873.187098  2278.759874
10         Oranges, Mandarines     14  2003.818886  2292.474998
2         Cassava and products     14

In [ ]:
"""
The Prophet model was successfully fitted across all crops and produced an average MAE of about 1,558 kg/ha and RMSE of about 1,722 kg/ha. This means that, on average, Prophet’s yield predictions were off by roughly 1.6–1.7 tonnes per hectare. Compared with ARIMA, Prophet performed slightly worse, suggesting that the yield series may not have strong structural trend breaks that Prophet is designed to capture. However, Prophet still performed better than SARIMA,
"""

## Models stacking

Stacking -combines all the predictions in different models to make one better prediction
Each model makes predictions on different things and we only stack strong models:
* ARIMA → trend  
* Prophet → flexible trend  
* Random Forest → features  
* XGBoost → best features model  
* SARIMA (drop it – weak)


### Collect predictions per crop

In [ ]:
# ============================================================
# STACKING PREP — COLLECT ALL MODEL PREDICTIONS
# ============================================================

stack_results = []

for crop in fao_final_df['item'].unique():
    
    df_crop = fao_final_df[fao_final_df['item'] == crop].copy()
    df_crop = df_crop.dropna(subset=['yield_kg_per_ha'])
    df_crop = df_crop.sort_values('year')
    
    if len(df_crop) < 8:
        continue
    
    train = df_crop[df_crop['year'] <= 2020]
    test = df_crop[df_crop['year'] > 2020]
    
    if len(test) == 0:
        continue
    
    # -----------------------------
    # TARGET
    # -----------------------------
    y_true = test['yield_kg_per_ha'].values
    
    # -----------------------------
    # XGBOOST
    # -----------------------------
    X_train = train[features]
    X_test = test[features]
    
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    
    model_xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6)
    model_xgb.fit(X_train, train['yield_kg_per_ha'])
    pred_xgb = model_xgb.predict(X_test)
    
    # -----------------------------
    # RANDOM FOREST
    # -----------------------------
    model_rf = RandomForestRegressor(n_estimators=200)
    model_rf.fit(X_train, train['yield_kg_per_ha'])
    pred_rf = model_rf.predict(X_test)
    
    # -----------------------------
    # ARIMA
    # -----------------------------
    model_arima = ARIMA(train['yield_kg_per_ha'], order=(1,1,1)).fit()
    pred_arima = model_arima.forecast(steps=len(test))
    
    # -----------------------------
    # PROPHET
    # -----------------------------
    df_prophet = train[['year', 'yield_kg_per_ha']].copy()
    df_prophet['ds'] = pd.to_datetime(df_prophet['year'].astype(str) + '-12-31')
    df_prophet['y'] = df_prophet['yield_kg_per_ha']
    
    model_prophet = Prophet()
    model_prophet.fit(df_prophet[['ds', 'y']])
    
    future = pd.DataFrame({
        'ds': pd.to_datetime(test['year'].astype(str) + '-12-31')
    })
    
    forecast = model_prophet.predict(future)
    pred_prophet = forecast['yhat'].values
    
    # -----------------------------
    # STACKING (simple average)
    # -----------------------------
    pred_stack = (
        pred_xgb + pred_rf + pred_arima + pred_prophet
    ) / 4
    
    # -----------------------------
    # METRICS
    # -----------------------------
    mae = mean_absolute_error(y_true, pred_stack)
    rmse = np.sqrt(mean_squared_error(y_true, pred_stack))
    
    stack_results.append({
        'crop': crop,
        'mae': mae,
        'rmse': rmse
    })

stack_df = pd.DataFrame(stack_results).sort_values('mae')

print("===== STACKED MODEL RESULTS =====")
print(stack_df)

print("\n===== OVERALL STACK PERFORMANCE =====")
print("Average MAE:", stack_df['mae'].mean())
print("Average RMSE:", stack_df['rmse'].mean())

07:13:21 - cmdstanpy - INFO - Chain [1] start processing
07:13:29 - cmdstanpy - INFO - Chain [1] done processing
07:13:30 - cmdstanpy - INFO - Chain [1] start processing
07:13:30 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
07:13:31 - cmdstanpy - INFO - Chain [1] start processing
07:13:31 - cmdstanpy - INFO - Chain [1] done processing
07:13:32 - cmdstanpy - INFO - Chain [1] start processing
07:13:33 - cmdstanpy - INFO - Chain [1] done processing
07:13:33 - cmdstanpy - INFO - Chain [1] start processing
07:13:34 - cmdstanpy - INFO - Chain [1] done processing
07:13:35 - cmdstanpy - INFO - Chain [1] start processing
07:13:35 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\AbigaelKariuki\anaconda3\envs\learn_env2\Lib\site-pack

===== STACKED MODEL RESULTS =====
                          crop          mae         rmse
4          Coffee and products    56.417166    59.785428
7           Maize and products    95.894676   109.577961
0                      Bananas   141.757772   186.490605
3        Coconuts - Incl Copra   272.249819   284.215106
5                   Groundnuts   297.268420   327.918461
8          Millet and products   305.832013   310.607809
14        Sorghum and products   329.042231   355.854301
9                       Onions   657.470547   927.417438
17          Wheat and products   706.409639   718.282241
13           Rice and products   726.467315   727.773244
12       Potatoes and products   847.537335   884.254175
1          Barley and products   917.747187   923.507790
10         Oranges, Mandarines  1026.238971  1028.608319
2         Cassava and products  1397.060160  1796.176427
15              Sweet potatoes  1435.453715  1438.006927
11     Pineapples and products  1862.638449  2664.0303

| Model         | MAE        | RMSE        |
| ------------- | ---------- | ----------- |
| ARIMA         | ~1382      | ~1510       |
| Prophet       | ~1558      | ~1722       |
| Random Forest | ~1177      | ~1308       |
| XGBoost       | ~1024      | ~1197       |
| **STACKED**   | **~968     | **~1108 ** |

** Stacking gives the BEST performance across all models
** Each model was good at something:
1. ARIMA → trends
2. Prophet → flexible trends
3. Random Forest → feature relationships
4. XGBoost → strongest feature model
5. Stacking - combines all strengths → reduces weaknesses

--The stacked model provides the best overall performance, achieving an average MAE of approximately 968 kg/ha and RMSE of approximately 1,108 kg/ha, outperforming all individual models. By combining predictions from time-series models (ARIMA and Prophet) and machine learning models (Random Forest and XGBoost), the stacked model captures both yield trends and key drivers more effectively. This shows that combining models leads to more accurate and stable predictions across different crops.
